## Compare FGCM and AuxTel/Spectractor atmospheric parameters

- author : Sylvie Dagoret-Campagne
- creation date : 2026-07-02

Compare the atmospheric parameters retrieved by **FGCM** (from the LSSTCam DRP visit table,
see `03-ReadFGCMAtmosphericParameters.ipynb`) with the atmospheric parameters retrieved by
**AuxTel/Spectractor** (from the spectroscopic fits, see `95_LoadAndConvertSpectroAuxtelFile.ipynb`):

| parameter | FGCM column | AuxTel/Spectractor column | quantity |
|---|---|---|---|
| PWV      | `pwv`   | `PWV [mm]_rum`      | Precipitable water vapor [mm] |
| Ozone    | `o3`    | `ozone [db]_rum`    | Ozone column [DU] |
| Aerosol depth | `tau` | `VAOD_rum`        | (Vertical) aerosol optical depth |
| Aerosol slope | `alpha` | `angstrom_exp_rum` | Angstrom exponent |

One figure is produced per atmospheric parameter. On each figure:
- the bottom x-axis is **MJD**,
- the top x-axis shows the corresponding **calendar date (YYYY-MM-DD)**,
- the night-by-night astronomical-night segmentation (gray vertical bands) is kept exactly
  as implemented in `03-ReadFGCMAtmosphericParameters.ipynb`,
- FGCM points are colored by `physical_filter` (small dots),
- AuxTel points are overplotted with error bars, colored by `FILTER` (larger markers).


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import pandas as pd
from astropy.table import Table
from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, get_sun
import astropy.units as u

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found -> interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found -> %matplotlib inline")

## Configuration

In [ ]:
# -- publication-style rc params (improved presentation) -----------------------
mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 12,
        "axes.labelsize": "large",
        "axes.titlesize": "large",
        "xtick.labelsize": "medium",
        "ytick.labelsize": "medium",
        "legend.fontsize": "medium",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.linewidth": 1.1,
    }
)

In [ ]:
# Rubin-LSST / Cerro Pachon
lsst = EarthLocation(lat=-30.2417 * u.deg, lon=-70.7366 * u.deg, height=2663 * u.m)

In [ ]:
# where the figures are saved
pathfigs = "figs_FGCM04_CompareFGCM_Auxtel"
prefix = "fgcm04"
if not os.path.exists(pathfigs):
    os.makedirs(pathfigs)
figtype = ".png"

In [ ]:
# ----- FGCM dataset (same collection / file as in 03-ReadFGCMAtmosphericParameters.ipynb) -----
REPO_URI = "dp2_prep"
collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

filename_fgcm = "data_fgcm/fgcm_rdp2_prep_cLSSTCam_runs_DRP_DP2_v30_0_0_DM-53881_stage1_20260701_134941.fits"

suptitle = f"repo {REPO_URI}, coll = {collection[0]} ..."

In [ ]:
# ----- AuxTel / Spectractor dataset (same file as in 95_LoadAndConvertSpectroAuxtelFile.ipynb) -----
PATH_DIRDATA_AUXTEL = "data_auxtelspectro"
PATH_FILEDATA_AUXTEL = (
    "keep_auxtel_atmosphere_feb26_gaiaspec_gaiatarget_calspecthroughput_filteredtightcuts.parquet"
)
filename_auxtel = os.path.join(PATH_DIRDATA_AUXTEL, PATH_FILEDATA_AUXTEL)

In [ ]:
# Mapping between FGCM parameters and AuxTel/Spectractor parameters
PARAM_MAP = {
    "pwv": {
        "fgcm_col": "pwv",
        "auxtel_col": "PWV [mm]_rum",
        "auxtel_err_col": "PWV [mm]_err_rum",
        "label": "PWV [mm]",
        "title": "Precipitable Water Vapor",
    },
    "o3": {
        "fgcm_col": "o3",
        "auxtel_col": "ozone [db]_rum",
        "auxtel_err_col": "ozone [db]_err_rum",
        "label": "O3 [DU]",
        "title": "Ozone column",
    },
    "tau": {
        "fgcm_col": "tau",
        "auxtel_col": "VAOD_rum",
        "auxtel_err_col": "VAOD_err_rum",
        "label": r"$\tau$ (aerosol optical depth)",
        "title": "Aerosol optical depth",
    },
    "alpha": {
        "fgcm_col": "alpha",
        "auxtel_col": "angstrom_exp_rum",
        "auxtel_err_col": "angstrom_exp_err_rum",
        "label": r"$\alpha$ (Angstrom exponent)",
        "title": "Aerosol slope (Angstrom exponent)",
    },
}

In [ ]:
# Colors for the FGCM points, by physical_filter (same palette as in 03-...)
default_filter_colors = {
    "u_24": "tab:blue",
    "g_6": "tab:green",
    "r_57": "tab:red",
    "i_39": "tab:orange",
    "z_20": "tab:gray",
    "y_10": "black",
}

# Colors/markers for the AuxTel points, by FILTER (holoscope configuration)
auxtel_style = {
    "empty": {"color": "darkviolet", "marker": "s"},
    "OG550_65mm_1": {"color": "saddlebrown", "marker": "^"},
}

## Tools

Astronomical-night segmentation (gray bands), kept identical in spirit to
`03-ReadFGCMAtmosphericParameters.ipynb`, but expressed directly in MJD since the
main x-axis of the comparison figures is MJD (with a calendar-date axis on top).

In [ ]:
def night_astronomical_mjd(day_mjd, location):
    """
    Return the (start_mjd, end_mjd) of the astronomical night (sun altitude < -18 deg)
    for the given site and starting MJD day. Returns (None, None) if no such night is found.
    """
    t_start = Time(day_mjd, format="mjd")
    t_grid = t_start + np.arange(0, 1.5, 5 / 1440)  # 1.5 day to fully capture the night
    altaz = AltAz(obstime=t_grid, location=location)
    sun_alt = get_sun(t_grid).transform_to(altaz).alt
    mask_night = sun_alt < -18 * u.deg
    night_mjds = t_grid.mjd[mask_night]
    if len(night_mjds) == 0:
        return None, None
    return night_mjds[0], night_mjds[-1]


def add_night_shading(ax, mjd_min, mjd_max, location):
    """Draw gray axvspan bands for each astronomical night between mjd_min and mjd_max."""
    start_day = int(np.floor(mjd_min))
    end_day = int(np.ceil(mjd_max))
    for day_mjd in np.arange(start_day, end_day, 1):
        start_night, end_night = night_astronomical_mjd(day_mjd, location)
        if start_night is not None:
            ax.axvspan(start_night, end_night, color="gray", alpha=0.08, zorder=0)

In [ ]:
def make_date_axis(ax_mjd, mjd_min, mjd_max, n_ticks=10):
    """Add a secondary top x-axis showing calendar dates (YYYY-MM-DD), aligned with the MJD axis."""
    ax_date = ax_mjd.twiny()
    ax_date.set_xlim(ax_mjd.get_xlim())
    tick_mjds = np.linspace(mjd_min, mjd_max, n_ticks)
    tick_labels = [Time(m, format="mjd", scale="utc").strftime("%Y-%m-%d") for m in tick_mjds]
    ax_date.set_xticks(tick_mjds)
    ax_date.set_xticklabels(tick_labels, rotation=45, ha="left", fontsize=9)
    ax_date.set_xlabel("Date (UTC)", fontsize=11, labelpad=8)
    return ax_date

In [ ]:
def get_fgcm_mjd(t_fgcm):
    """Return an array of MJD values for the FGCM table, robust to schema variants."""
    if "expMidptMJD" in t_fgcm.colnames:
        return np.asarray(t_fgcm["expMidptMJD"], dtype=float)
    # fallback: derive MJD from the expMidpt column (isot strings, possibly bytes)
    raw = t_fgcm["expMidpt"]
    raw_str = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in raw])
    return Time(raw_str, format="isot", scale="utc").mjd

In [ ]:
def plot_atm_parameter_comparison(t_fgcm, df_auxtel, param_key, savefig=True):
    """
    Compare a single atmospheric parameter between FGCM and AuxTel/Spectractor.

    Parameters
    ----------
    t_fgcm : astropy.table.Table
        FGCM visit table (must contain 'physical_filter' and the FGCM parameter column).
    df_auxtel : pandas.DataFrame
        AuxTel/Spectractor dataframe (must contain 'MJD', 'FILTER' and the parameter/err columns).
    param_key : str
        One of the keys of PARAM_MAP ('pwv', 'o3', 'tau', 'alpha').
    """
    cfg = PARAM_MAP[param_key]

    # ---- FGCM side ----
    mjd_fgcm = get_fgcm_mjd(t_fgcm)
    y_fgcm = np.asarray(t_fgcm[cfg["fgcm_col"]], dtype=float)
    filt_fgcm = np.asarray(t_fgcm["physical_filter"])
    mask_fgcm = np.isfinite(mjd_fgcm) & np.isfinite(y_fgcm)

    # ---- AuxTel side ----
    mjd_aux = df_auxtel["MJD"].to_numpy(dtype=float)
    y_aux = df_auxtel[cfg["auxtel_col"]].to_numpy(dtype=float)
    yerr_aux = df_auxtel[cfg["auxtel_err_col"]].to_numpy(dtype=float)
    band_aux = df_auxtel["FILTER"].to_numpy()
    mask_aux = np.isfinite(mjd_aux) & np.isfinite(y_aux)

    mjd_min = np.nanmin([mjd_fgcm[mask_fgcm].min(), mjd_aux[mask_aux].min()])
    mjd_max = np.nanmax([mjd_fgcm[mask_fgcm].max(), mjd_aux[mask_aux].max()])

    fig, ax = plt.subplots(figsize=(18, 8))

    # night-by-night astronomical-night shading (kept as in 03-...)
    add_night_shading(ax, mjd_min, mjd_max, lsst)

    # FGCM points, colored by physical_filter
    for f, c in default_filter_colors.items():
        m = mask_fgcm & (filt_fgcm == f)
        if np.sum(m) > 0:
            ax.scatter(
                mjd_fgcm[m],
                y_fgcm[m],
                s=10,
                alpha=0.5,
                color=c,
                label=f"FGCM {f}",
                zorder=2,
            )

    # AuxTel points, colored by FILTER, with error bars
    for band, style in auxtel_style.items():
        m = mask_aux & (band_aux == band)
        if np.sum(m) > 0:
            ax.errorbar(
                mjd_aux[m],
                y_aux[m],
                yerr=yerr_aux[m],
                fmt=style["marker"],
                ms=6,
                mfc=style["color"],
                mec="black",
                mew=0.5,
                ecolor=style["color"],
                elinewidth=1.0,
                capsize=2,
                alpha=0.85,
                linestyle="none",
                label=f"AuxTel {band}",
                zorder=3,
            )

    ax.set_xlim(mjd_min - 0.5, mjd_max + 0.5)
    ax.set_xlabel("MJD")
    ax.set_ylabel(cfg["label"])
    ax.set_title(f"{cfg['title']}: FGCM vs AuxTel/Spectractor\nGray = astronomical night at LSST site")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Dataset / filter", ncol=2, framealpha=0.9)

    make_date_axis(ax, mjd_min, mjd_max, n_ticks=10)

    fig.suptitle(suptitle, y=1.06, fontsize=10)
    fig.tight_layout()

    if savefig:
        figname = os.path.join(pathfigs, f"{prefix}_compare_{param_key}{figtype}")
        fig.savefig(figname, bbox_inches="tight")

    plt.show()
    return fig, ax

## Load data

### FGCM

In [ ]:
t_fgcm = Table.read(filename_fgcm)
print(t_fgcm.colnames)
print(t_fgcm[:2])

In [ ]:
unique_filters_fgcm = np.unique(t_fgcm["physical_filter"])
print(unique_filters_fgcm)

### AuxTel / Spectractor

In [ ]:
df_auxtel = pd.read_parquet(filename_auxtel)
print(df_auxtel.shape)
df_auxtel[["MJD", "FILTER", "PWV [mm]_rum", "ozone [db]_rum", "VAOD_rum", "angstrom_exp_rum"]].head()

In [ ]:
unique_bands_auxtel = np.unique(df_auxtel["FILTER"])
print(unique_bands_auxtel)

## Plots: FGCM vs AuxTel/Spectractor, one figure per atmospheric parameter

### PWV

In [ ]:
_ = plot_atm_parameter_comparison(t_fgcm, df_auxtel, "pwv")

### Ozone (O3)

In [ ]:
_ = plot_atm_parameter_comparison(t_fgcm, df_auxtel, "o3")

### Aerosol optical depth (tau)

In [ ]:
_ = plot_atm_parameter_comparison(t_fgcm, df_auxtel, "tau")

### Aerosol slope / Angstrom exponent (alpha)

In [ ]:
_ = plot_atm_parameter_comparison(t_fgcm, df_auxtel, "alpha")